In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("used_cars_cleaned_07.csv")

# Prepare features and target from your current cleaned dataset
X = df.drop(columns=["price_gbp", "condition"], errors="ignore")
y = df["price_gbp"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# All learned preprocessing is fitted on the training subset
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

# Train and predict
linear_model.fit(X_train, y_train)

train_predictions = linear_model.predict(X_train)
val_predictions = linear_model.predict(X_val)

# Compare training and validation performance
results = []

for name, actual, predicted in [
    ("Training", y_train, train_predictions),
    ("Validation", y_val, val_predictions)
]:
    results.append({
        "Dataset": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted)
    })

display(pd.DataFrame(results).round(3))

# Inspect some validation predictions
comparison = pd.DataFrame({
    "Actual Price": y_val,
    "Predicted Price": val_predictions
})

comparison["Absolute Error"] = (
    comparison["Actual Price"] - comparison["Predicted Price"]
).abs()

display(comparison.head(10).round(2))

print("Negative validation predictions:", (val_predictions < 0).sum())

,Dataset,MAE (£),RMSE (£),R²
0,Training,1623.392,8036.429,0.942
1,Validation,2382.688,30518.424,0.591


,Actual Price,Predicted Price,Absolute Error
78752,17860,17091.94,768.06
20722,17100,17577.81,477.81
187958,17995,19477.86,1482.86
276571,15998,17610.76,1612.76
261948,11295,13182.15,1887.15
242167,31788,38278.84,6490.84
343980,38490,38977.70,487.70
165892,25097,25498.79,401.79
190705,15915,14591.80,1323.20
287712,2395,3735.78,1340.78


Negative validation predictions: 1915


In [3]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(solver="lsqr"))
])

param_grid = {
    "model__alpha": [0.1, 1, 10, 100, 1000]
}

cv = KFold(n_splits=3, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    verbose=1,
    refit=True
)

# Tune using training data only
search.fit(X_train, y_train)

best_ridge = search.best_estimator_

print("Best parameters:", search.best_params_)
print("Best cross-validation MAE (£):", round(-search.best_score_, 2))

# Evaluate the selected model on validation data
predictions = best_ridge.predict(X_val)

print("\nValidation results")
print("MAE (£):", round(mean_absolute_error(y_val, predictions), 2))
print("RMSE (£):", round(np.sqrt(
    mean_squared_error(y_val, predictions)
), 2))
print("R²:", round(r2_score(y_val, predictions), 4))

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best parameters: {'model__alpha': 0.1}
Best cross-validation MAE (£): 2370.8

Validation results
MAE (£): 2361.28
RMSE (£): 32369.49
R²: 0.5401


In [4]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Exclude missing-value indicators from polynomial expansion
poly_cols = [
    col for col in numeric_cols
    if not col.endswith("_missing")
]

indicator_cols = [
    col for col in numeric_cols
    if col.endswith("_missing")
]

polynomial_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler())
])

poly_preprocessor = ColumnTransformer([
    ("numeric_poly", polynomial_pipeline, poly_cols),
    ("indicators", "passthrough", indicator_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

poly_model = Pipeline([
    ("preprocessor", poly_preprocessor),
    ("model", Ridge(alpha=100, solver="lsqr"))
])

# Use the same split as previous models for a fair comparison
poly_model.fit(X_train, y_train)

results = []

for name, X_part, y_part in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val)
]:
    predictions = poly_model.predict(X_part)

    results.append({
        "Dataset": name,
        "MAE (£)": mean_absolute_error(y_part, predictions),
        "RMSE (£)": np.sqrt(mean_squared_error(y_part, predictions)),
        "R²": r2_score(y_part, predictions)
    })

display(pd.DataFrame(results).round(3))

,Dataset,MAE (£),RMSE (£),R²
0,Training,3736.620,22454.474,0.547
1,Validation,3873.204,40692.884,0.273


In [6]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

param_grid = {
    "preprocessor__numeric_poly__polynomial__degree": [1, 2],
    "model__alpha": [0.1, 1, 10, 100, 1000]
}

cv = KFold(n_splits=3, shuffle=True, random_state=42)

poly_search = GridSearchCV(
    estimator=poly_model,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    pre_dispatch=2,
    verbose=1,
    refit=True,
    error_score="raise"
)

# Tune on training data only
poly_search.fit(X_train, y_train)

best_poly_model = poly_search.best_estimator_

print("Best parameters:", poly_search.best_params_)
print("Best CV MAE (£):", round(-poly_search.best_score_, 2))

# Compare training and validation performance
results = []

for name, X_part, y_part in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val)
]:
    predictions = best_poly_model.predict(X_part)

    results.append({
        "Dataset": name,
        "MAE (£)": mean_absolute_error(y_part, predictions),
        "RMSE (£)": np.sqrt(mean_squared_error(y_part, predictions)),
        "R²": r2_score(y_part, predictions)
    })

display(pd.DataFrame(results).round(3))

# Show all parameter combinations, best first
tuning_results = pd.DataFrame(poly_search.cv_results_)
tuning_results["CV MAE (£)"] = -tuning_results["mean_test_score"]

display(
    tuning_results[[
        "param_preprocessor__numeric_poly__polynomial__degree",
        "param_model__alpha",
        "CV MAE (£)",
        "rank_test_score"
    ]]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters: {'model__alpha': 0.1, 'preprocessor__numeric_poly__polynomial__degree': 1}
Best CV MAE (£): 2372.05


,Dataset,MAE (£),RMSE (£),R²
0,Training,1819.036,8307.615,0.938
1,Validation,2361.069,32369.424,0.540


,param_preprocessor__numeric_poly__polynomial__degree,param_model__alpha,CV MAE (£),rank_test_score
0,1,0.1,2372.054102,1
1,2,1.0,2431.797711,2
2,1,1.0,2556.612167,3
3,2,0.1,3034.019190,4
4,2,10.0,3126.221097,5
5,1,10.0,3469.187274,6
6,2,100.0,4002.347206,7
7,2,1000.0,4736.164435,8
8,1,100.0,4791.599733,9
9,1,1000.0,6088.757380,10


In [7]:
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Use the polynomial degree selected by your previous search
sgd_model = Pipeline([
    ("preprocessor", best_poly_model.named_steps["preprocessor"]),
    ("model", SGDRegressor(
        loss="squared_error",
        penalty="l2",
        max_iter=3000,
        tol=1e-4,
        average=True,
        random_state=42
    ))
])

param_grid = {
    "model__alpha": [0.0001, 0.001, 0.01],
    "model__learning_rate": ["adaptive"],
    "model__eta0": [0.0001, 0.001]
}

sgd_search = GridSearchCV(
    sgd_model,
    param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    pre_dispatch=2,
    verbose=1
)

sgd_search.fit(X_train, y_train)

predictions = sgd_search.predict(X_val)

print("Best parameters:", sgd_search.best_params_)
print("Validation MAE (£):",
      round(mean_absolute_error(y_val, predictions), 2))
print("Validation RMSE (£):",
      round(np.sqrt(mean_squared_error(y_val, predictions)), 2))
print("Validation R²:", round(r2_score(y_val, predictions), 4))

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best parameters: {'model__alpha': 0.0001, 'model__eta0': 0.001, 'model__learning_rate': 'adaptive'}
Validation MAE (£): 4004.54
Validation RMSE (£): 40274.38
Validation R²: 0.288


In [8]:
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

log_ridge = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", Ridge(solver="lsqr"))
    ]),
    func=np.log1p,
    inverse_func=np.expm1
)

log_search = GridSearchCV(
    log_ridge,
    param_grid={
        "regressor__model__alpha": [0.01, 0.1, 1, 10, 100]
    },
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    pre_dispatch=2,
    verbose=1
)

log_search.fit(X_train, y_train)

# Predictions are already in pounds
predictions = log_search.predict(X_val)

print("Best parameters:", log_search.best_params_)
print("Validation MAE (£):",
      round(mean_absolute_error(y_val, predictions), 2))
print("Validation RMSE (£):",
      round(np.sqrt(mean_squared_error(y_val, predictions)), 2))
print("Validation R²:", round(r2_score(y_val, predictions), 4))

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best parameters: {'regressor__model__alpha': 0.1}
Validation MAE (£): 1722.45
Validation RMSE (£): 37244.65
Validation R²: 0.3911


In [9]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Work on copies of the existing split
X_train_text = X_train.copy()
X_val_text = X_val.copy()

for frame in [X_train_text, X_val_text]:
    frame["version"] = frame["version"].fillna("").astype(str)

numeric_cols = X_train_text.select_dtypes(
    include="number"
).columns.tolist()

# Exclude version: it will be handled by CountVectorizer
other_categorical_cols = [
    col for col in X_train_text.select_dtypes(
        include=["object", "string", "category"]
    ).columns
    if col != "version"
]

text_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),

    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), other_categorical_cols),

    # A string selector passes a 1D text column to CountVectorizer
    ("version_text", CountVectorizer(
        lowercase=True,
        strip_accents="unicode",
        token_pattern=r"(?u)\b\w[\w.-]*\b",
        min_df=3,
        max_features=15000
    ), "version")
])

text_model = Pipeline([
    ("preprocessor", text_preprocessor),
    ("model", Ridge(solver="lsqr"))
])

text_search = GridSearchCV(
    text_model,
    param_grid={
        "preprocessor__version_text__ngram_range": [(1, 1), (1, 2)],
        "model__alpha": [0.01, 0.1, 1, 10]
    },
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    pre_dispatch=2,
    verbose=1
)

text_search.fit(X_train_text, y_train)

predictions = text_search.predict(X_val_text)

print("Best parameters:", text_search.best_params_)
print("Validation MAE (£):",
      round(mean_absolute_error(y_val, predictions), 2))
print("Validation RMSE (£):",
      round(np.sqrt(mean_squared_error(y_val, predictions)), 2))
print("Validation R²:", round(r2_score(y_val, predictions), 4))

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best parameters: {'model__alpha': 1, 'preprocessor__version_text__ngram_range': (1, 2)}
Validation MAE (£): 2618.29
Validation RMSE (£): 34017.9
Validation R²: 0.492


In [10]:
err = (y_val - best_ridge.predict(X_val)).abs().sort_values(ascending=False)
X_val.loc[err.index[:20]].assign(actual=y_val, err=err)

,make,model,version,mileage_miles,reg_year,fuel,transmission,power_hp,body_type,body_color,...,previous_owners,drivetrain,emission_class,annual_tax_gbp,seller_type,seller_city,seller_county,co2_gmile,actual,err
84760,Ferrari,F40,2.9 Coupe 2dr Petrol Manual (478 bhp),8500,1992,Petrol,Manual,0,Coupe,Unknown,...,4,Rear Wheel Drive,Unknown,445.0,Trade,Preston,Lancashire,620,9999999,7.813809e+06
198368,Ferrari,275,GTB/4,27029,1968,Petrol,Manual,0,Coupe,Red,...,4,Unknown,Unknown,375.0,Trade,Swindon,Wiltshire,322,3349990,3.042426e+06
113848,Rimac,Nevera,120kWh GT Auto 4WD 2dr,735,2023,Electric,Automatic,1888,Coupe,Green,...,0,Four Wheel Drive,Unknown,200.0,Trade,London,London,0,1500000,1.361962e+06
301467,Ferrari,599,6.0 V12 GTO F1 Euro 5 2dr,8833,2011,Petrol,Automatic,661,Coupe,Red,...,4,Rear Wheel Drive,Euro 5,790.0,Trade,Horsham,West Sussex,661,1299995,1.017777e+06
339895,Alfa Romeo,Giulietta,SPRINT ZAGATO. 1 OF 200. EX COMPETITION CAR. E...,7315,1960,Petrol,Manual,0,Coupe,White,...,2,Unknown,Unknown,230.0,Trade,Pulborough,West Sussex,216,395995,4.335557e+05
344988,Rolls-Royce,Ghost,6.6 V12 Auto Euro 6 4dr,22000,2015,Petrol,Automatic,563,Saloon,Blue,...,4,Rear Wheel Drive,Euro 6,790.0,Private,BARNSLEY,SOUTH YORKSHIRE,526,420000,3.198684e+05
66137,Lamborghini,Aventador,6.5 V12 LP 750-4 Superveloce Roadster ISR 4WD ...,10500,2017,Petrol,Automatic,750,Convertible,Orange,...,3,Four Wheel Drive,Euro 6,790.0,Trade,Huddersfield,West Yorkshire,595,899995,2.835655e+05
40886,Mercedes-Benz,G Class,4.0 G63 V8 BiTurbo MHEV AMG Carbon Edition Spd...,60,2026,Petrol Hybrid,Automatic,597,SUV,Green,...,1,Four Wheel Drive,Euro 6,200.0,Trade,Ascot,Berkshire,542,559995,2.797964e+05
71873,Mercedes-Benz,SLS,6.2 63 V8 AMG GT SpdS DCT Euro 5 2dr,6900,2013,Petrol,Automatic,583,Coupe,Grey,...,7,Rear Wheel Drive,Euro 5,790.0,Trade,Wirral,Merseyside,496,499995,2.681680e+05
328792,Lamborghini,Aventador,6.5 V12 LP 780-4 Ultimae Roadster ISR 4WD Euro...,2700,2022,Petrol,Automatic,769,Convertible,Orange,...,1,Four Wheel Drive,Euro 6,200.0,Trade,London,Unknown,803,729995,2.672433e+05


In [12]:
from sklearn.base import clone
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

svd_linear = Pipeline([
    ("preprocessor", clone(text_preprocessor)),
    ("reduction", TruncatedSVD(random_state=42)),
    ("model", LinearRegression())
])

# Dimensionality reduction can be expensive: start small
svd_search = GridSearchCV(
    svd_linear,
    param_grid={
        "reduction__n_components": [50, 100, 200]
    },
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    verbose=1,
    error_score="raise"
)

svd_search.fit(X_train_text, y_train)

best_svd_model = svd_search.best_estimator_
predictions = best_svd_model.predict(X_val_text)

print("Best parameters:", svd_search.best_params_)
print("Validation MAE (£):",
      round(mean_absolute_error(y_val, predictions), 2))
print("Validation RMSE (£):",
      round(np.sqrt(mean_squared_error(y_val, predictions)), 2))
print("Validation R²:", round(r2_score(y_val, predictions), 4))

variance = (
    best_svd_model.named_steps["reduction"]
    .explained_variance_ratio_.sum()
)
print("Feature variance retained:", f"{variance:.1%}")

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best parameters: {'reduction__n_components': 100}
Validation MAE (£): 6905.43
Validation RMSE (£): 43595.43
Validation R²: 0.1657
Feature variance retained: 80.1%
